# LLM Chatbot using LangChain

This notebook builds a conversational chatbot using **LangChain** and an **LLM**.

### Features
- Uses LangChain's modern message/runnable APIs
- Conversation memory with `InMemoryChatMessageHistory`
- Supports OpenAI-compatible chat models
- Simple interactive chat loop
- Environment-variable based API key handling

> Install the required packages and set your API key before running the chatbot.

In [ ]:
# Install dependencies
%pip install -q -U langchain langchain-openai langchain-core python-dotenv

## 1. Import libraries and configure the API key

Create an environment variable named `OPENAI_API_KEY`.

For example, in a `.env` file:

```text
OPENAI_API_KEY=your_api_key_here
```

Do not hard-code or share your API key in the notebook.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not set. Add it to your environment or .env file."
    )

print("API key loaded successfully.")

## 2. Create the LLM

The example below uses `ChatOpenAI`. You can change the model to another model available to your API account.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
)

print("LLM initialized.")

## 3. Create the chatbot prompt

The system message defines the chatbot's behavior. Human and AI messages are then passed to the model.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful, knowledgeable and friendly AI assistant. "
        "Answer clearly and accurately. If you are uncertain, say so."
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm
print("Chat chain created.")

## 4. Add conversation memory

`InMemoryChatMessageHistory` stores messages during the current Python session. This gives the chatbot context from earlier messages.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()

def chat(user_input: str) -> str:
    """Send a message to the chatbot while preserving conversation history."""
    response = chain.invoke({
        "history": chat_history.messages,
        "input": user_input,
    })

    chat_history.add_user_message(user_input)
    chat_history.add_ai_message(response.content)

    return response.content

## 5. Test the chatbot

In [ ]:
response = chat("Hello! My name is Faisal. Can you explain what an LLM is?")
print("Assistant:", response)

## 6. Test conversational memory

The chatbot should remember information from the previous message in this session.

In [ ]:
response = chat("What is my name?")
print("Assistant:", response)

## 7. Interactive chatbot

Run this cell and type messages. Enter `exit`, `quit`, or `bye` to stop.

In [ ]:
print("🤖 LangChain LLM Chatbot")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in {"exit", "quit", "bye"}:
        print("Assistant: Goodbye!")
        break

    if not user_input:
        continue

    try:
        answer = chat(user_input)
        print(f"Assistant: {answer}\n")
    except Exception as e:
        print(f"Error: {e}\n")

## 8. Clear conversation memory

Run this cell whenever you want to start a fresh conversation.

In [ ]:
chat_history.clear()
print("Conversation history cleared.")

## 9. Inspect the conversation history

This is useful for understanding how LangChain stores the conversation context.

In [ ]:
for message in chat_history.messages:
    print(f"{message.type}: {message.content}")

## How the chatbot works

```text
User Input
    ↓
ChatPromptTemplate
    ↓
Conversation History + New Message
    ↓
LLM (ChatOpenAI)
    ↓
AI Response
    ↓
Conversation History
```

### Important
- The memory in this notebook is **temporary** and is lost when the Python process/kernel is restarted.
- For production applications, use persistent storage such as a database or a dedicated LangChain history backend.
- API calls can incur usage charges depending on your provider and account.